# 101 Spark basics

The goal of this lab is to get familiar with Spark programming.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [ ]:
# Do this only once in lab machines to install PySpark
pip install pyspark

In [ ]:
# Do this in lab machines to fix configuration issues
import os
os.environ.pop("SPARK_HOME", None)

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

## 101-1 Spark warm-up

Load the ```capra``` and ```divinacommedia``` datasets and try the following actions:
- Show their content (```collect```)
- Count their rows (```count```)
- Split phrases into words (```map``` or ```flatMap```; what’s the difference?)
- Check the results (remember: evaluation is lazy)
- Try the ```toDebugString``` function to check the execution plan
    - In PySpark, use ```toDebugString().decode("unicode_escape")```

In [ ]:
rddCapra = sc.textFile("../../../../datasets/capra.txt")
rddDC = sc.textFile("../../../../datasets/divinacommedia.txt")

In [ ]:
rddCapraWords1 = rddCapra.map(lambda x : x.split(" ") )
rddCapraWords1.collect()

In [ ]:
rddCapraWords1.count()

In [ ]:
rddCapraWords2 = rddCapra.flatMap(lambda x : x.split(" ") )
rddCapraWords2.collect()

In [ ]:
rddCapraWords2.count()

In [ ]:
rddL = rddCapra. \
   flatMap(lambda x : x.split(" ") ). \
   map(lambda x : (x,1)). \
   reduceByKey(lambda x,y : x+y)
print(rddL.toDebugString().decode("unicode_escape"))

## 101-2 Basic Spark jobs

Implement on Spark the following jobs and test them on both capra and divinacommedia datasets.

- **Word count**: count the number of occurrences of each word
  - Result: (sopra, 1), (la, 4), …
- **Word length count**: count the number of occurrences of words of given lengths
  - Result: (2, 4), (5, 8)
- Count the average length of words given their first letter (i.e., words that begin with "s" have an average length of 5)
  - Result: (s, 5), (l, 2), …
- Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...

Also, check how sorting works and try to sort key-value RDDs by descending values.

In [ ]:
# Word count
rddCapra. \
  flatMap(lambda x : x.split(" ") ). \
  map(lambda x : (x,1)). \
  reduceByKey(lambda x,y : x + y). \
  map(lambda kv: (kv[1],kv[0])). \
  sortByKey(False). \
  collect()

In [ ]:
# Word length count
rddCapra. \
  flatMap(lambda x : x.split(" ") ). \
  map(lambda x : (len(x),1)). \
  reduceByKey(lambda x,y : x + y). \
  collect()